# Analyze and save Raman spectra

In [ ]:
from pathlib import Path

import arcadia_pycolor as apc
import numpy as np
import pandas as pd
import plotly.colors as pcolors
import plotly.express as px
import plotly.graph_objects as go
import ramanspy as rp
from plotly.subplots import make_subplots
from scipy.optimize import curve_fit

from raman_analysis.utils import collect_spectra_from_dir, plot_spectra_in_dir

apc.plotly.setup()
%load_ext autoreload
%autoreload 2

raman_chem_perturbation_dir = Path("data/raman-chemical-perturbations")
raman_ph_perturbation_dir = Path("data/raman-ph-perturbations")
plot_dir_htmls = Path("plots/Raman/html")
plot_dir_svg = Path("plots/Raman/svg")



## Chemical perturbations


### Colormap


In [ ]:
chem_perturbation_palette = apc.Palette(
    "chem_perturbations", [apc.taupe, apc.vital, apc.aster, apc.lime]
)
chem_perturbation_palette

In [ ]:
def generate_subtract_dict_chem(samples):
    """Generate a dictionary mapping sample names to their buffer names for subtraction."""
    subtract_dict = {}
    for name in samples:
        date = name.split("/")[0]
        buffer = name.split(" in ")[-1]
        subtract_dict[name] = f"{date}/{buffer}"
    return subtract_dict

### Load data and pre-processing pipeline

In [ ]:
data_dir = raman_chem_perturbation_dir

data_list, sample_df = collect_spectra_from_dir(data_dir, "instantraman", recursive=True)

pipeline_chem = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(350, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        rp.preprocessing.baseline.ModPoly(poly_order=9),
    ]
)


### Utility function to print sample table

In [ ]:
plotted_samples = {} # Track all plotted samples for raw data
df_sample_tables = []
def print_sample_table(df, subtract_dict):
    proteins = []
    buffers = []
    for k,v in subtract_dict.items():
        protein_filename = df.index[df['Name'] == k].tolist()[0]
        proteins.append(protein_filename)
        if v is None:
            buffers.append(None)
            continue
        buffer_filename = df.index[df['Name'] == v].tolist()[0]
        buffers.append(buffer_filename)
    df = pd.DataFrame({
        "Protein experiment": list(subtract_dict.keys()),
        "Protein filename": proteins,
        "Buffer filename": buffers
    })
    return df


### All available proteins

In [ ]:

buffers = ["Tris", "Tris+TCEP", "Tris+EDTA", "Tris+TCEP+EDTA"]
all_samples = sample_df["Name"].str.split("/").str[-1].str.split(" in ").str[0].unique()
all_samples = [s for s in all_samples if not any(b in s for b in buffers)]
all_samples

### Plot spectra for BSA, FAF-BSA, DNAse, Cytochrome C

In [ ]:
xranges = {
    "BSA [10 mgml]": [None, [360, 700], [1150, 1378]],
    "FAF-BSA [10 mgml]": [None, [360, 700], [1150, 1378]],
    "DNAse [12.5 mgml]": [None, [360, 700], [1150, 1378]],
    "CytC [10 mgml]": [None, [360, 700], [1500, 1650]],
}
yranges = {
    "BSA [10 mgml]": [None, [-200, 800], [-500, 2500]],
    "FAF-BSA [10 mgml]": [None, [-200, 800], [-500, 2500]],
    "DNAse [12.5 mgml]": [None, [-275, 915], None],
    "CytC [10 mgml]": [None, None, None],
}

# First plot is full spectrum, the second and third are zooms
fig_widths = ["full_wide", "half_square", "half_square"]

n_groups = sample_df["Group ID"].nunique()

buffer_dict = {
    "BSA [10 mgml]": buffers[:-1], # TCEP+EDTA excluded — causes precipitation in BSA
    "FAF-BSA [10 mgml]": buffers[:-1], # TCEP+EDTA excluded — causes precipitation in BSA
    "DNAse [12.5 mgml]": buffers,
    "CytC [10 mgml]": buffers,
}


for protein in xranges.keys():
    for i, (xrange, yrange) in enumerate(zip(xranges[protein], yranges[protein], strict=True)):
        # include files that are the protein in the corresponding buffer
        include_samples = [
            n
            for n in sample_df["Name"]
            if Path(n).name.startswith(protein)
            and any(n.endswith(buf) for buf in buffer_dict[protein])
        ]
        subtract_id = generate_subtract_dict_chem(include_samples)
        include_samples += list(subtract_id.values())
        fig, names, rp_container = plot_spectra_in_dir(
            data_list,
            col_to_plot="Intensity (dark-subtracted)",
            data_dir=data_dir,
            subtract_id=subtract_id,
            rp_pipeline=pipeline_chem,
            colors=chem_perturbation_palette,
            include=include_samples,
            xaxis_title="Wavenumber (cm⁻¹)",
            yaxis_title="Intensity (a.u.)",
            title=f"{protein} | {xrange}_cm-1 | {"_".join(buffer_dict[protein])}",
        )
        fig.update_xaxes(range=xrange)
        fig.update_yaxes(range=yrange)
        fig.data = tuple(trace for trace in fig.data if trace.name not in subtract_id.values())
        fig.update_layout(showlegend=False)
        apc.plotly.style_plot(fig, monospaced_axes="all")
        apc.plotly.set_figure_dimensions(fig, size=fig_widths[i])
        protein_dir_svg = plot_dir_svg / protein
        protein_dir_html = plot_dir_htmls / protein
        protein_dir_svg.mkdir(parents=True, exist_ok=True)
        protein_dir_html.mkdir(parents=True, exist_ok=True)

        fig.write_image(protein_dir_svg / f"{protein}_{xrange}_cm-1.svg")
        fig.write_html(protein_dir_html / f"{protein}_{xrange}_cm-1.html", include_plotlyjs="cdn")
        fig.show()
    df_sample_tables.append(print_sample_table(sample_df, subtract_id))

### Plot spectra for additional proteins (not in main figures)

In [ ]:
additional_proteins = [s for s in all_samples if s not in xranges.keys()]
for protein in additional_proteins:
    include_samples = [
        n
        for n in sample_df["Name"]
        if Path(n).name.startswith(protein)
        and any(n.endswith(buf) for buf in buffers)
    ]
    subtract_id = generate_subtract_dict_chem(include_samples)
    include_samples += list(subtract_id.values())
    fig, names, rp_container = plot_spectra_in_dir(
        data_list,
        col_to_plot="Intensity (dark-subtracted)",
        data_dir=data_dir,
        subtract_id=subtract_id,
        rp_pipeline=pipeline_chem,
        colors=chem_perturbation_palette,
        include=include_samples,
        xaxis_title="Wavenumber (cm⁻¹)",
        yaxis_title="Intensity (a.u.)",
        title=protein,
    )

    df_sample_tables.append(print_sample_table(sample_df, subtract_id))

    fig.data = tuple(trace for trace in fig.data if trace.name not in subtract_id.values())

    apc.plotly.style_plot(fig, monospaced_axes="all")
    apc.plotly.set_figure_dimensions(fig, size="full_wide")

    protein_dir_svg = plot_dir_svg / protein
    protein_dir_html = plot_dir_htmls / protein
    protein_dir_svg.mkdir(parents=True, exist_ok=True)
    protein_dir_html.mkdir(parents=True, exist_ok=True)

    fig.write_image(protein_dir_svg / f"{protein}_{xrange}_cm-1.svg")
    fig.write_html(protein_dir_html / f"{protein}_{xrange}_cm-1.html", include_plotlyjs="cdn")
    fig.show()

### Plot spectra for buffers

In [ ]:
buffers

In [ ]:
include_samples = [n for n in sample_df["Name"] if any(n == f"2025-12-10/{buf}" for buf in buffers)]

fig, names, rp_container = plot_spectra_in_dir(
    data_list,
    col_to_plot="Intensity (dark-subtracted)",
    data_dir=data_dir,
    subtract_id=None,
    rp_pipeline=pipeline_chem,
    colors=chem_perturbation_palette,
    include=include_samples,
    xaxis_title="Wavenumber (cm⁻¹)",
    yaxis_title="Intensity (a.u.)",
    title="Buffers",
)

subtract_id = {k:None for k in include_samples}
df_sample_tables.append(print_sample_table(sample_df, subtract_id))

apc.plotly.style_plot(fig, monospaced_axes="all")
apc.plotly.set_figure_dimensions(fig, size="full_wide")

fig.write_image(plot_dir_svg / "buffers.svg")
fig.write_html(plot_dir_htmls / "buffers.html", include_plotlyjs="cdn")
fig.show()

## pH perturbations

### Colormap and constants

In [ ]:
ph_fig_width = 1300
ph_fig_height = 400
ph_gradient = apc.gradients.red_blue
BSA_color = apc.aster
BSA_TCEP_color = apc.amber

ph_to_plot = [0.3, 1, 1.3, 2, 2.3, 3, 7.5, 8, 9, 9.5, 12, 12.7]
ph_traces_to_show = ph_to_plot # modify to show only a subset of traces
ph_vals_to_show_on_colorbar = [0.3, 7.5, 12.7]

Phe_wavenumber = 1002 # Phenylalanine peak, used for normalization

### Load data and pre-processing pipeline

In [ ]:
data_list, sample_df = collect_spectra_from_dir(
    raman_ph_perturbation_dir, "instantraman", recursive=True
)

pipeline_pH = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(400, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        rp.preprocessing.baseline.ModPoly(poly_order=6),
    ]
)

n_groups = sample_df["Group ID"].nunique()

In [ ]:
def generate_subtract_dict_pH(_replicate, sample_string):
    """ Make dictionary mapping sample names to pH-matched buffer names. """
    ph_vals = [0, 0.3, 1, 1.3, 2, 2.3, 3, 3.3, 4, 5, 6,
                7.5, 8, 9, 9.5, 10, 10.7, 11, 11.7, 12,
                12.7, 13, 13.7, 14]
    ph_vals_str = [str(p) for p in ph_vals]

    sample_names = sample_df["Name"].unique().tolist()
    all_protein = [n for n in sample_names if sample_string + " pH" in n and _replicate in n]

    subtraction_key = "TCEP" if "TCEP" in sample_string else "buffers"
    subtract_dict = {}
    for p in all_protein:
        for ph in ph_vals_str:
            if f"{sample_string} pH {ph}" in p:
                subtract_dict[p] = [
                    b for b in sample_names if b.endswith(f"{_replicate}/{subtraction_key} pH {ph}")
                ][0]
    # For non-TCEP data, subtract pH=7.5 data instead of matching pH to account for
    # peak shifts in the buffer solutions that are not seen in the protein solutions
    for k, v in list(subtract_dict.items()):
        if "TCEP" not in v:
            if any(v.endswith(f"pH {ph}") for ph in ["8", "9", "9.5"]):
                new_v = v.rsplit(" ", 1)[0] + " 7.5"
                subtract_dict[k] = new_v
    return subtract_dict

### Plot Raman spectra of BSA under different pH conditions

In [ ]:
replicate = "rep1" # replicate ID to plot
ph_to_plot_labels = [f"pH {ph_val}" for ph_val in ph_to_plot]

spectral_regions = {
    (1400, 1420): "Carboxylate",
    (1647, 1657): "Amide I<br>⍺-Helix",
    (1328, 1341): "Trp,<br>C⍺-H def",
    (940, 950): "C⍺-C stretch",
}

sample_names_dict = {}
sample_containers_dict = {}

for sample in ["BSA", "BSA+TCEP"]:
    subtract_id = generate_subtract_dict_pH(replicate, sample)
    subtract_id = {
        k: v
        for k, v in subtract_id.items()
        if any(k.endswith(label) for label in ph_to_plot_labels)
    }

    include_samples = list(subtract_id.keys()) + list(subtract_id.values())
    exclude_from_plot = subtract_id.values()

    colors = pcolors.sample_colorscale(px.colors.qualitative.Dark24,n_groups)

    fig, names, rp_container = plot_spectra_in_dir(
        data_list,
        col_to_plot="Intensity (dark-subtracted)",
        data_dir=raman_ph_perturbation_dir,
        subtract_id=subtract_id,
        rp_pipeline=pipeline_pH,
        colors=colors,
        include=include_samples,
        height=ph_fig_height,
        width=ph_fig_width,
        xaxis_title="Wavenumber (cm⁻¹)",
        yaxis_title="Intensity (a.u.)",
        title=f"{sample} pH perturbation",
    )
    fig.data = tuple(
        trace for trace in fig.data if not any(trace.name == exclude for exclude in exclude_from_plot)
    )
    for trace in fig.data:
        if not any(str(trace.name).endswith(f"pH {ph}") for ph in ph_traces_to_show):
            trace.visible = "legendonly"

    gradient = ph_gradient.resample_as_palette(len(fig.data))
    for i, trace in enumerate(fig.data):
        trace.line.color = gradient[i]

    # Add a colorbar for the gradient with pH values
    colorbar_trace = go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale=[[i / (len(gradient) - 1), color] for i, color in enumerate(gradient)],
            cmin=min(ph_to_plot),
            cmax=max(ph_to_plot),
            colorbar=dict(
                title="pH",
                tickvals=ph_to_plot,
                ticktext=[str(ph) for ph in ph_to_plot],
                lenmode="pixels",
                len=200,
            ),
            color=[min(ph_to_plot), max(ph_to_plot)],
            showscale=True,
            size=0.1,
        ),
        showlegend=False,
        hoverinfo="none",
    )
    # pH color bar
    colorbar_trace.marker.colorbar.update(
        x=1.0,
        y=1.05,
        xanchor="left",
        yanchor="bottom",
        orientation="h",
        thickness=5,
        lenmode="pixels",
        len=200,
        title_side="top",
        tickvals=ph_vals_to_show_on_colorbar,
        ticktext=[str(ph) for ph in ph_vals_to_show_on_colorbar],
    )
    fig.add_trace(colorbar_trace)

    # Spectral region bars: x in data coords (moves with zoom), y in paper coords (stays at top)
    for (x0, x1), label in spectral_regions.items():
        fig.add_shape(
            type="line",
            x0=x0,
            x1=x1,
            y0=1.04,
            y1=1.04,
            xref="x",
            yref="paper",
            line=dict(color="black", width=2),
            layer="above",
        )
        fig.add_annotation(
            x=(x0 + x1) / 2,
            y=1.06,
            xref="x",
            yref="paper",
            text=label,
            showarrow=False,
            font=dict(size=10),
            yanchor="bottom",
            xanchor="center",
        )

    sample_names_dict[sample] = names
    sample_containers_dict[sample] = rp_container
    df_sample_tables.append(print_sample_table(sample_df, subtract_id))

    fig.update_layout(margin=dict(t=100))
    apc.plotly.style_plot(fig, monospaced_axes="all")
    apc.plotly.set_figure_dimensions(fig, size="full_wide")
    fig.show()

    fig.write_image(plot_dir_svg / f"{sample}_ph_spectra.svg")
    fig.write_html(plot_dir_htmls / f"{sample}_ph_spectra.html", include_plotlyjs="cdn")


### Plot protein features as a function of pH

In [ ]:
def intensity_at(spectral_axis, spectral_data_row, wavenumber):
    idx = np.argmin(np.abs(spectral_axis - wavenumber))
    return float(spectral_data_row[idx])


def fit_gaussian_peak(spectral_axis, spectral_data_row, center, window=60):
    """Fit a Gaussian to a spectral region; return (center, fwhm). Returns (nan, nan) on failure."""
    mask = (spectral_axis >= center - window) & (spectral_axis <= center + window)
    x, y = spectral_axis[mask], spectral_data_row[mask]
    if len(x) < 4:
        return np.nan, np.nan

    def gaussian(x, amp, mu, sigma):
        return amp * np.exp(-((x - mu) ** 2) / (2 * sigma**2))

    try:
        p0 = [y.max(), center, window / 3]
        bounds = ([0, center - window, 1], [np.inf, center + window, window])
        popt, _ = curve_fit(gaussian, x, y, p0=p0, bounds=bounds, maxfev=5000)
        _, mu, sigma = popt
        fwhm = 2.355 * abs(sigma)
        return float(mu), float(fwhm)
    except RuntimeError:
        return np.nan, np.nan


peak_records = []
for sample in ["BSA", "BSA+TCEP"]:
    names = sample_names_dict[sample]
    container = sample_containers_dict[sample]
    ax = container.spectral_axis
    for i, name in enumerate(names):
        if "BSA" not in name:
            continue
        for ph in ph_to_plot:
            if not name.endswith(f"pH {ph}"):
                continue
            row = container.spectral_data[i]
            center_1657, fwhm_1657 = fit_gaussian_peak(ax, row, 1657)
            peak_records.append(
                {
                    "pH": ph,
                    "sample": sample,
                    "i1341": intensity_at(ax, row, 1341) / intensity_at(ax, row, Phe_wavenumber),
                    "i1657": intensity_at(ax, row, center_1657) / intensity_at(ax, row, Phe_wavenumber),
                    "i1657_center": center_1657,
                    "i1657_fwhm": fwhm_1657,
                    "i1412": intensity_at(ax, row, 1412) / intensity_at(ax, row, Phe_wavenumber),
                    "i945": intensity_at(ax, row, 945) / intensity_at(ax, row, Phe_wavenumber),
                }
            )

peak_df = pd.DataFrame(peak_records)

peak_params = [
    ("i1657", "Amide I peak", "Peak ratio"),
    ("i1657_center", "Amide I center", "Wavenumber (cm⁻¹)"),
    ("i1657_fwhm", "Amide I FWHM", "Wavenumber (cm⁻¹)"),
    ("i1341", "Trp, C⍺-H def (1,341 cm⁻¹)", "Peak ratio"),
    ("i945", "C⍺-C stretch (945 cm⁻¹)", "Peak ratio"),
    ("i1412", "Carboxylate (1,412 cm⁻¹)", "Peak ratio"),
]

n_cols = 3
n_rows = int(np.ceil(len(peak_params) / n_cols))
sample_colors = {"BSA": BSA_color, "BSA+TCEP": BSA_TCEP_color}

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[label for _, label, _ in peak_params],
    vertical_spacing=0.3,
    horizontal_spacing=0.15,
)
for idx, (col, _, ytitle) in enumerate(peak_params):
    row_idx, col_idx = divmod(idx, n_cols)
    for sample in ["BSA", "BSA+TCEP"]:
        df_s = peak_df[peak_df["sample"] == sample].sort_values("pH")
        df_low = df_s[df_s["pH"] <= 3]
        df_high = df_s[df_s["pH"] >= 7.5]
        color = sample_colors[sample]

        # Solid lines for each measured pH range
        for seg_idx, df_seg in enumerate([df_low, df_high]):
            fig.add_trace(
                go.Scatter(
                    x=df_seg["pH"],
                    y=df_seg[col],
                    mode="lines+markers",
                    name=sample,
                    legendgroup=sample,
                    showlegend=(idx == 0 and seg_idx == 0),
                    line=dict(color=color),
                ),
                row=row_idx + 1,
                col=col_idx + 1,
            )

        # Dashed connector bridging the unmeasured pH gap (3–7.5)
        fig.add_trace(
            go.Scatter(
                x=[df_low["pH"].iloc[-1], df_high["pH"].iloc[0]],
                y=[df_low[col].iloc[-1], df_high[col].iloc[0]],
                mode="lines",
                name=sample,
                legendgroup=sample,
                showlegend=False,
                line=dict(color=color, dash="dash"),
            ),
            row=row_idx + 1,
            col=col_idx + 1,
        )
        fig.update_yaxes(title_text=ytitle, row=row_idx + 1, col=col_idx + 1)

fig.update_layout(legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1))
fig.update_xaxes(title_text="pH")
apc.plotly.style_plot(fig, monospaced_axes="all")
apc.plotly.set_figure_dimensions(fig, size="full_wide")
fig.update_layout(height=500)
fig.show()

fig.write_image(plot_dir_svg / "peaks_vs_pH.svg")
fig.write_html(plot_dir_htmls / "peaks_vs_pH.html", include_plotlyjs="cdn")

## Write experiment -> raw data mapping table

In [ ]:
figure_ids = ["Figure " + str(i+1) for i in range(len(df_sample_tables))] # Placeholder: Finalize the IDs once the figure order is known

with open("experiment-sample-map.md", "w") as f:
    for fig_id, df in zip(figure_ids, df_sample_tables, strict=True):
        df_md = df.to_markdown(index=False)
        f.write(f"## {fig_id}\n\n")
        f.write(df_md)
        f.write("\n\n")
